In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch03-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "03-03-chemical-databases.py"

# 依資料層級核對 PDB 複合體與配體

預設讀取官方回傳快照，重現教材結果。若要即時查詢，在分析格設定 LIVE = True；連線失敗不會自動改用快照。圖是 CCD 結構的二維重畫，沒有顯示複合體座標。


In [ ]:
%%writefile "{script}"
from pathlib import Path
import os, json, hashlib, urllib.request
from datetime import datetime, timezone
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
# Genuine field projection of five official RCSB API responses, not generated measurements.
SNAPSHOT = {'entry': {'rcsb_id': '1IEP', 'struct': {'title': 'CRYSTAL STRUCTURE OF THE C-ABL KINASE DOMAIN IN COMPLEX WITH STI-571.'}, 'exptl': [{'crystals_number': 1, 'method': 'X-RAY DIFFRACTION'}], 'rcsb_entry_info': {'resolution_combined': [2.1]}, 'rcsb_entry_container_identifiers': {'assembly_ids': ['1', '2'], 'entity_ids': ['1', '2', '3', '4'], 'entry_id': '1IEP', 'model_ids': [1], 'non_polymer_entity_ids': ['2', '3'], 'polymer_entity_ids': ['1'], 'pubmed_id': 12154025, 'rcsb_id': '1IEP'}}, 'ligand': {'chem_comp': {'formula': 'C29 H31 N7 O', 'formula_weight': 493.603, 'id': 'STI', 'name': '4-(4-METHYL-PIPERAZIN-1-YLMETHYL)-N-[4-METHYL-3-(4-PYRIDIN-3-YL-PYRIMIDIN-2-YLAMINO)-PHENYL]-BENZAMIDE', 'pdbx_ambiguous_flag': 'N', 'pdbx_formal_charge': 0, 'pdbx_initial_date': '2001-04-11T00:00:00.000+00:00', 'pdbx_modified_date': '2020-06-17T00:00:00.000+00:00', 'pdbx_processing_site': 'RCSB', 'pdbx_release_status': 'REL', 'three_letter_code': 'STI', 'type': 'non-polymer'}, 'rcsb_chem_comp_descriptor': {'InChI': 'InChI=1S/C29H31N7O/c1-21-5-10-25(18-27(21)34-29-31-13-11-26(33-29)24-4-3-12-30-19-24)32-28(37)23-8-6-22(7-9-23)20-36-16-14-35(2)15-17-36/h3-13,18-19H,14-17,20H2,1-2H3,(H,32,37)(H,31,33,34)', 'InChIKey': 'KTUFNOKKBVMGRW-UHFFFAOYSA-N', 'SMILES': 'Cc1ccc(cc1Nc2nccc(n2)c3cccnc3)NC(=O)c4ccc(cc4)CN5CCN(CC5)C', 'SMILES_stereo': 'Cc1ccc(cc1Nc2nccc(n2)c3cccnc3)NC(=O)c4ccc(cc4)CN5CCN(CC5)C', 'comp_id': 'STI'}}, 'nonpoly2': {'rcsb_id': '1IEP_2', 'pdbx_entity_nonpoly': {'comp_id': 'CL', 'entity_id': '2', 'name': 'CHLORIDE ION'}, 'rcsb_nonpolymer_entity_container_identifiers': {'asym_ids': ['C', 'D', 'E', 'F', 'H', 'I'], 'auth_asym_ids': ['A', 'B'], 'chem_ref_def_id': 'CL', 'entity_id': '2', 'entry_id': '1IEP', 'nonpolymer_comp_id': 'CL', 'rcsb_id': '1IEP_2'}}, 'nonpoly3': {'rcsb_id': '1IEP_3', 'pdbx_entity_nonpoly': {'comp_id': 'STI', 'entity_id': '3', 'name': '4-(4-METHYL-PIPERAZIN-1-YLMETHYL)-N-[4-METHYL-3-(4-PYRIDIN-3-YL-PYRIMIDIN-2-YLAMINO)-PHENYL]-BENZAMIDE'}, 'rcsb_nonpolymer_entity_container_identifiers': {'asym_ids': ['G', 'J'], 'auth_asym_ids': ['A', 'B'], 'chem_ref_def_id': 'STI', 'entity_id': '3', 'entry_id': '1IEP', 'nonpolymer_comp_id': 'STI', 'rcsb_id': '1IEP_3'}}, 'protein': {'rcsb_id': '1IEP_1', 'rcsb_entity_source_organism': [{'ncbi_scientific_name': 'Mus musculus', 'ncbi_taxonomy_id': 10090}], 'rcsb_polymer_entity': {'formula_weight': 33.744, 'pdbx_description': 'PROTO-ONCOGENE TYROSINE-PROTEIN KINASE ABL', 'pdbx_ec': '2.7.1.112', 'pdbx_fragment': 'KINASE DOMAIN', 'pdbx_number_of_molecules': 2, 'rcsb_multiple_source_flag': 'N', 'rcsb_source_part_count': 1, 'rcsb_source_taxonomy_count': 1, 'src_method': 'man', 'rcsb_ec_lineage': [{'depth': 1, 'id': '2', 'name': 'Transferases'}, {'depth': 2, 'id': '2.7', 'name': 'Transferring phosphorus-containing groups'}, {'depth': 3, 'id': '2.7.1', 'name': 'Phosphotransferases with an alcohol group as acceptor'}, {'depth': 4, 'id': '2.7.1.112', 'name': 'protein-tyrosine kinase'}, {'depth': 1, 'id': '2', 'name': 'Transferases'}, {'depth': 2, 'id': '2.7', 'name': 'Transferring phosphorus-containing groups'}, {'depth': 3, 'id': '2.7.10', 'name': 'Protein-tyrosine kinases'}, {'depth': 4, 'id': '2.7.10.2', 'name': 'non-specific protein-tyrosine kinase'}], 'rcsb_macromolecular_names_combined': [{'name': 'PROTO-ONCOGENE TYROSINE-PROTEIN KINASE ABL', 'provenance_code': 'ECO:0000304', 'provenance_source': 'PDB Preferred Name'}, {'name': 'P150', 'provenance_code': 'ECO:0000303', 'provenance_source': 'PDB Synonym'}, {'name': 'C-ABL', 'provenance_code': 'ECO:0000303', 'provenance_source': 'PDB Synonym'}], 'rcsb_enzyme_class_combined': [{'depth': 4, 'ec': '2.7.1.112', 'provenance_source': 'PDB Primary Data'}, {'depth': 4, 'ec': '2.7.10.2', 'provenance_source': 'UniProt'}], 'rcsb_polymer_name_combined': {'names': ['Tyrosine-protein kinase ABL1'], 'provenance_source': 'UniProt Name'}}}}
PROVENANCE = {'entry': {'url': 'https://data.rcsb.org/rest/v1/core/entry/1IEP', 'retrieved_utc': '2026-11-04T07:53:31.449097+00:00', 'sha256': '09499268eccba586c0443425240eccc171748fd24bc964c448e792c2ad96e4c2'}, 'ligand': {'url': 'https://data.rcsb.org/rest/v1/core/chemcomp/STI', 'retrieved_utc': '2026-11-04T07:53:32.007793+00:00', 'sha256': '071661c3f0296df1a405b9157d5190d7cc0d50297c13b97a3ce2f7056007cc59'}, 'nonpoly2': {'url': 'https://data.rcsb.org/rest/v1/core/nonpolymer_entity/1IEP/2', 'retrieved_utc': '2026-11-04T07:54:30.691088+00:00', 'sha256': 'b16e3436b6f9294b3bac4728014da015c62a7e3c756b000fe24578fc0150f92d'}, 'nonpoly3': {'url': 'https://data.rcsb.org/rest/v1/core/nonpolymer_entity/1IEP/3', 'retrieved_utc': '2026-11-04T07:54:31.111869+00:00', 'sha256': '0b6ba6bf33771208e8d47b975856c143eb8ae0673d5bdf73c3919b68fa99b83a'}, 'protein': {'url': 'https://data.rcsb.org/rest/v1/core/polymer_entity/1IEP/1', 'retrieved_utc': '2026-11-04T07:54:32.027582+00:00', 'sha256': 'de9dd22ddc7cf736d5dc6151eeee97c918404ccebaaf8bc7ad0769bc172d1b01'}}
FIELDS = {'entry': ['rcsb_id', 'struct', 'exptl', 'rcsb_entry_info', 'rcsb_entry_container_identifiers'], 'ligand': ['chem_comp', 'rcsb_chem_comp_descriptor'], 'nonpoly2': ['rcsb_id', 'pdbx_entity_nonpoly', 'rcsb_nonpolymer_entity_container_identifiers'], 'nonpoly3': ['rcsb_id', 'pdbx_entity_nonpoly', 'rcsb_nonpolymer_entity_container_identifiers'], 'protein': ['rcsb_id', 'rcsb_entity_source_organism', 'rcsb_polymer_entity']}

def project(name, response):
    result = {key: response[key] for key in FIELDS[name]}
    if name == 'entry':
        result['rcsb_entry_info'] = {'resolution_combined': result['rcsb_entry_info']['resolution_combined']}
    if name == 'protein':
        result['rcsb_entity_source_organism'] = [
            {'ncbi_scientific_name': item['ncbi_scientific_name'], 'ncbi_taxonomy_id': item['ncbi_taxonomy_id']}
            for item in result['rcsb_entity_source_organism']]
    return result

# Default: reproduce the captured teaching data. Live mode has no silent fallback.
LIVE = os.environ.get('BOOK_CH03_LIVE', '0') == '1'
records = SNAPSHOT
if LIVE:
    records = {}
    for name, info in PROVENANCE.items():
        with urllib.request.urlopen(info['url'], timeout=45) as response:
            records[name] = project(name, json.load(response))
    print('MODE: live API', datetime.now(timezone.utc).isoformat())
else:
    print('MODE: captured API snapshot', PROVENANCE['entry']['retrieved_utc'])
entry = records['entry']
ligand = records['ligand']
nonpolymers = [records['nonpoly2'], records['nonpoly3']]
link = next(item for item in nonpolymers if item['pdbx_entity_nonpoly']['comp_id'] == 'STI')
assert entry['rcsb_id'] == '1IEP'
assert link['pdbx_entity_nonpoly']['entity_id'] in entry['rcsb_entry_container_identifiers']['non_polymer_entity_ids']
assert link['pdbx_entity_nonpoly']['comp_id'] == ligand['chem_comp']['id']
method = entry['exptl'][0]['method']
resolution = entry['rcsb_entry_info']['resolution_combined'][0]
organisms = [item['ncbi_scientific_name'] for item in records['protein']['rcsb_entity_source_organism']]
smiles = ligand['rcsb_chem_comp_descriptor']['SMILES_stereo']
mol = Chem.MolFromSmiles(smiles)
assert mol is not None
formula = rdMolDescriptors.CalcMolFormula(mol)
assert formula == ligand['chem_comp']['formula'].replace(' ', '')
print('entry:', entry['rcsb_id'], 'entity:', link['pdbx_entity_nonpoly']['entity_id'], 'component:', ligand['chem_comp']['id'])
print('method:', method, 'resolution_A:', resolution, 'protein source:', organisms)
print('component formula:', formula, 'heavy atoms:', mol.GetNumHeavyAtoms())
print('nonpolymer components:', [item['pdbx_entity_nonpoly']['comp_id'] for item in nonpolymers])
if not LIVE:
    assert method == 'X-RAY DIFFRACTION' and resolution == 2.1
    assert organisms == ['Mus musculus']
    assert mol.GetNumHeavyAtoms() == 37
folder=Path('../../figures') if Path('../../_quarto.yml').is_file() else Path('figures')
folder.mkdir(exist_ok=True)
fig, axes=plt.subplots(1,2,figsize=(11,4.4),gridspec_kw={'width_ratios':[1.2,1]})
axes[0].imshow(Draw.MolToImage(mol,size=(650,400)))
axes[0].axis('off'); axes[0].set_title('STI: chemical component definition')
axes[1].axis('off')
labels=[f"PDB entry {entry['rcsb_id']}\n{method} | {resolution:.2f} Å",f"Nonpolymer entity {link['rcsb_id']}\ncomponent ID = STI",f"CCD component STI\n{formula} | {mol.GetNumHeavyAtoms()} heavy atoms"]
for y,label in zip([.83,.5,.17],labels):
    axes[1].text(.5,y,label,ha='center',va='center',fontsize=11,bbox=dict(boxstyle='round,pad=.7',fc='#eef6f7',ec='#0f766e'))
for top,bottom in [(.72,.61),(.39,.28)]:
    axes[1].annotate('',xy=(.5,bottom),xytext=(.5,top),arrowprops=dict(arrowstyle='->',color='#0f766e',lw=2))
fig.tight_layout();fig.savefig(folder/'03-03-chemical-databases.png',dpi=160);plt.close(fig)


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "03-03-chemical-databases.png")))